In [17]:


import re, json, random
from collections import Counter
from typing import List, Dict, Set
import torch
import sentencepiece as spm

CKPT_PATH = "checkpoints/best_step8000_val3.862.pt"
SPM_PATH  = "tokenizer/ka_sp_24000.model"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [18]:
from src.models import GPT, GPTConfig

print(f"Using device: {DEVICE}")
print(f"SPM_MODEL: {SPM_PATH}")
print(f"CKPT_PATH: {CKPT_PATH}")

ckpt = torch.load(CKPT_PATH, map_location="cpu")
gcfg = GPTConfig(**ckpt["config"])
model = GPT(gcfg).to(DEVICE)
miss = model.load_state_dict(ckpt["model"], strict=True)
print("Loaded model. Missing/unexpected keys:", miss)
print("Model config:", gcfg)

sp = spm.SentencePieceProcessor(model_file=SPM_PATH)
BOS, EOS, UNK = sp.bos_id(), sp.eos_id(), sp.unk_id()
VOCAB = sp.vocab_size()
print(f"Loaded SentencePiece; vocab_size = {VOCAB} (UNK id={UNK})")

Using device: cuda
SPM_MODEL: tokenizer/ka_sp_24000.model
CKPT_PATH: checkpoints/best_step8000_val3.862.pt
Loaded model. Missing/unexpected keys: <All keys matched successfully>
Model config: GPTConfig(vocab_size=24000, n_layer=12, n_head=8, d_model=512, n_ctx=512, ffn_mult=4, dropout=0.0, rope_theta=10000.0, tie_weights=True, bias=False, use_sdpa=True, grad_checkpoint=False)
Loaded SentencePiece; vocab_size = 24000 (UNK id=0)


In [19]:
GEORGIAN_ONLY_DECODE = True

GE_PIECE_RE = re.compile(r"^▁?[ა-ჰ]+(?:[-– ][ა-ჰ]+)?$")

def build_georgian_vocab_mask():
    mask = torch.full((VOCAB,), float("-inf"))
    space_ok = sp.piece_to_id("▁")
    if space_ok >= 0:
        mask[space_ok] = 0.0
    for tid in range(VOCAB):
        piece = sp.id_to_piece(tid)
        if GE_PIECE_RE.match(piece):
            mask[tid] = 0.0
    # allowing EOS
    if EOS is not None and EOS >= 0:
        mask[EOS] = 0.0
    return mask.to(DEVICE)

GE_MASK = build_georgian_vocab_mask() if GEORGIAN_ONLY_DECODE else None


In [20]:
FEW_SHOT_PRIMER = (
    "უპასუხე მოკლედ, ფაქტობრივად, ქართულად.\n"
    "სომხეთის დედაქალაქია — ერევანი.\n"
    "თურქეთის დედაქალაქია — ანკარა.\n"
    "საქართველოს ოფიციალური ენაა — ქართული.\n"
)
USE_PRIMER = True

def with_primer(q: str) -> str:
    return (FEW_SHOT_PRIMER + q) if USE_PRIMER else q

In [21]:
@torch.no_grad()
def generate(
    prompt: str,
    max_new_tokens: int = 12,
    min_new_tokens: int = 2,
    temperature: float = 0.1,    
    top_p: float = 0.9,
    top_k: int = 0,
    rep_penalty: float = 1.20,
    no_repeat_ngram_size: int = 4,
    stop_on_eos: bool = False,
    stop_at_punct: bool = True,
    add_bos: bool = True,
) -> str:
    ids = sp.encode(prompt, out_type=int, add_bos=add_bos, add_eos=False)
    x = torch.tensor(ids, dtype=torch.long, device=DEVICE)[None, :]
    start = x.size(1)

    seen = {}
    if no_repeat_ngram_size > 0 and x.size(1) >= no_repeat_ngram_size:
        for i in range(x.size(1) - no_repeat_ngram_size + 1):
            gram = tuple(x[0, i:i+no_repeat_ngram_size].tolist())
            seen[gram] = seen.get(gram, 0) + 1

    def update_seen(seq, n=no_repeat_ngram_size):
        if n <= 0 or seq.size(1) < n:
            return
        gram = tuple(seq[0, -n:].tolist())
        seen[gram] = seen.get(gram, 0) + 1

    def violates_no_repeat_ngram(next_id, seq, n=no_repeat_ngram_size):
        if n <= 0 or seq.size(1) < n - 1:
            return False
        tail = tuple(seq[0, -(n - 1):].tolist()) if n > 1 else tuple()
        cand = tail + (int(next_id.item()),)
        return seen.get(cand, 0) > 0

    PUNCTS = [".", "!", "?", "…", "։", "።"]

    for t in range(max_new_tokens):
        logits, _ = model(x)
        logits = logits[:, -1, :]

        # forbidding <unk>
        if UNK is not None and UNK >= 0:
            logits[:, UNK] = -1e9

        
        if GE_MASK is not None:
            logits = logits + GE_MASK

        
        for tok in ["▁ეს", "▁არის", "▁და", "▁რომ"]:
            tid = sp.piece_to_id(tok)
            if tid >= 0:
                logits[:, tid] /= rep_penalty

       
        if rep_penalty != 1.0:
            for tid in set(x[0].tolist()):
                logits[:, tid] /= rep_penalty

        
        if temperature <= 1e-6:
            next_id = logits.argmax(dim=-1)
        else:
            logits = logits / temperature
            if top_k > 0:
                v, ix = torch.topk(logits, top_k)
                keep = torch.full_like(logits, -float("inf"))
                keep.scatter_(1, ix, v)
                logits = keep
            if top_p < 1.0:
                sorted_logits, sorted_idx = torch.sort(logits, descending=True)
                probs = torch.softmax(sorted_logits, dim=-1)
                cumprobs = torch.cumsum(probs, dim=-1)
                cutoff = (cumprobs > top_p).float().argmax(dim=-1, keepdim=True)
                mask = torch.arange(probs.size(1), device=probs.device)[None, :] > cutoff
                sorted_logits[mask] = -float("inf")
                logits = torch.zeros_like(logits).scatter(1, sorted_idx, sorted_logits)
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).squeeze(1)

        
        if no_repeat_ngram_size > 0 and violates_no_repeat_ngram(next_id, x):
            next_id = logits.argmax(dim=-1)

        x = torch.cat([x, next_id[:, None]], dim=1)
        update_seen(x)

        
        if t + 1 >= min_new_tokens:
            if stop_on_eos and next_id.item() == EOS:
                break
            if stop_at_punct:
                cont = sp.decode(x[0, start:].tolist()).strip()
                if len(cont) > 0 and cont[-1:] in PUNCTS:
                    break

    return sp.decode(x[0, start:].tolist())

In [22]:
GE_LETTERS = r"[ა-ჰ]"

def extract_answer(s: str) -> str:
    s = s.strip()
    s = re.sub(r"[«»“”\"'—–\-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = re.split(r"[.!?…։።]", s)[0].strip()
    s = re.sub(r"^(ეს\s+არის|ეს\s+არის,\s*რომ|რომ)\s+", "", s, flags=re.IGNORECASE)
    m = re.search(fr"({GE_LETTERS}{{2,}}(?:\s+{GE_LETTERS}{{2,}})?)", s)
    return m.group(1) if m else s

def norm(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r"[“”\"'«»„.,!?;:()\[\]{}\-–—~…]", "", s)
    s = re.sub(r"\s+", " ", s)
    return s

def match_alias(pred: str, aliases: Set[str]) -> bool:
    p = norm(pred)
    alias_norm = [norm(a) for a in aliases]
    return any(p == a or a in p or p in a for a in alias_norm)

TESTS: List[Dict] = [
    {
        "name": "capital_of_georgia",
        "paraphrases": [
            "საქართველოს დედაქალაქია — ",
            "საქართველოს დედაქალაქია: ",
            "საქართველოს დედაქალაქია ___.",
            "რომელი ქალაქია საქართველოს დედაქალაქი? მოკლე პასუხი.",
        ],
        "answers": {"თბილისი", "ქ. თბილისი", "tbilisi", "თბილის"},
    },
    {
        "name": "currency_of_georgia",
        "paraphrases": [
            "საქართველოს ვალუტაა — ",
            "საქართველოს ეროვნული ვალუტაა: ",
            "საქართველოს ფულის ერთეული — ",
        ],
        "answers": {"ლარი", "GEL", "ლარი (GEL)", "lari"},
    },
    {
        "name": "official_language",
        "paraphrases": [
            "საქართველოს ოფიციალური ენაა — ",
            "რა ენაა სახელმწიფო საქართველოში? მხოლოდ ერთი სიტყვა.",
            "საქართველოს სახელმწიფო ენა — ",
        ],
        "answers": {"ქართული", "ქართული ენა"},
    },
    {
        "name": "author_vefxistqavosani",
        "paraphrases": [
            "„ვეფხისტყაოსნის“ ავტორია — ",
            "ვინ დაწერა „ვეფხისტყაოსანი“? მოკლე პასუხი.",
            "ავტორი: ვეფხისტყაოსანი — ",
        ],
        "answers": {"შოთა რუსთაველი", "რუსთაველი", "შ. რუსთაველი"},
    },
    {
        "name": "author_eugene_onegin",
        "paraphrases": [
            "ვინ დაწერა „ევგენი ონეგინი“? მოკლე პასუხი.",
            "„ევგენი ონეგინის“ ავტორია — ",
            "რომელია „ევგენი ონეგინის“ ავტორი? ერთი სიტყვით.",
        ],
        "answers": {"ალექსანდრე პუშკინი", "ალექსანდრ პუშკინი", "პუშკინი",
                    "ალექსანდრ სერგეევის ძე პუშკინი", "알ექსანდრე სერგეევის ძე პუშკინი".replace("알", "ა")},
    },
    {
        "name": "anthem_of_georgia",
        "paraphrases": [
            "საქართველოს ჰიმნია — ",
            "რა ჰქვია საქართველოს ეროვნულ ჰიმნს? მოკლე პასუხი.",
            "საქართველოს ეროვნული ჰიმნი — ",
        ],
        "answers": {"თავისუფლება", "tavisupleba"},
    },
    {
        "name": "continent_region",
        "paraphrases": [
            "საქართველო მდებარეობს — ",
            "რომელ რეგიონში მდებარეობს საქართველო? მოკლე პასუხი.",
            "ეპასუხე მხოლოდ ერთი სიტყვით: საქართველოს მდებარეობა",
        ],
        "answers": {"კავკასიაში", "კავკასია", "ევრაზიაში", "ევრაზია"},
    },
    {
        "name": "largest_city",
        "paraphrases": [
            "საქართველოს უდიდესი ქალაქია — ",
            "რომელია საქართველოს ყველაზე დიდი ქალაქი? მოკლე პასუხი.",
            "საქართველოს ყველაზე დიდ ქალაქს ეწოდება — ",
        ],
        "answers": {"თბილისი", "ქ. თბილისი", "tbilisi"},
    },
]


In [23]:
def ask(q: str) -> str:
    q2 = with_primer(q)
    raw = generate(
        prompt=q2,
        max_new_tokens=10,
        min_new_tokens=2,
        temperature=0.1,
        top_p=0.9,
        top_k=0,
        rep_penalty=1.2,
        no_repeat_ngram_size=4,
        stop_on_eos=False,
        stop_at_punct=True,
        add_bos=True,
    )
    return extract_answer(raw)

In [24]:
def evaluate_tests(tests: List[Dict], stability_samples: int = 0) -> Dict:
    random.seed(1234)
    total_prompts, exact_hits = 0, 0
    per_task_hits = {}
    paraphrase_consistency = {}
    examples = []

    for t in tests:
        name = t["name"]
        aliases = t["answers"]
        preds_raw, preds_norm = [], []

        for q in t["paraphrases"]:
            pred = ask(q)
            preds_raw.append(pred)
            preds_norm.append(norm(pred))
            ok = match_alias(pred, aliases)
            exact_hits += int(ok)
            total_prompts += 1
            examples.append({"task": name, "prompt": q, "pred": pred, "ok": bool(ok)})

        per_task_hits[name] = any(match_alias(p, aliases) for p in preds_raw)

        if preds_norm:
            mode_ans, cnt = Counter(preds_norm).most_common(1)[0]
            paraphrase_consistency[name] = cnt / len(preds_norm)

        if stability_samples > 0 and t["paraphrases"]:
            q0 = t["paraphrases"][0]
            reps = [norm(ask(q0)) for _ in range(stability_samples)]
            mode_rep, cnt_rep = Counter(reps).most_common(1)[0]
            paraphrase_consistency[name] = (
                paraphrase_consistency[name] + (cnt_rep / stability_samples)
            ) / 2.0

    overall_prompt_exact = exact_hits / total_prompts if total_prompts else 0.0
    overall_task_solved = sum(per_task_hits.values()) / len(tests) if tests else 0.0
    overall_consistency = (
        sum(paraphrase_consistency.values()) / len(paraphrase_consistency)
        if paraphrase_consistency else 0.0
    )

    report = {
        "overall_prompt_exact": round(overall_prompt_exact, 4),
        "overall_task_solved": round(overall_task_solved, 4),
        "overall_paraphrase_consistency": round(overall_consistency, 4),
        "task_hits": per_task_hits,
        "task_consistency": {k: round(v, 3) for k, v in paraphrase_consistency.items()},
        "examples": examples[:20],
    }
    return report

In [25]:
for s in [
    "საქართველოს დედაქალაქია — ",
    "საქართველოს ვალუტაა — ",
    "„ვეფხისტყაოსნის“ ავტორია — ",
    "ვინ დაწერა „ევგენი ონეგინი“? მოკლე პასუხი.",
]:
    print("Prompt:", s)
    print("→", ask(s))
    print("-" * 60)


Prompt: საქართველოს დედაქალაქია — 
→ საქართველო და
------------------------------------------------------------
Prompt: საქართველოს ვალუტაა — 
→ სნაიპერიული კურსით
------------------------------------------------------------
Prompt: „ვეფხისტყაოსნის“ ავტორია — 
→ საუკუნის მეორე
------------------------------------------------------------
Prompt: ვინ დაწერა „ევგენი ონეგინი“? მოკლე პასუხი.
→ ამ ეტაპზე
------------------------------------------------------------


In [26]:
TESTS: List[Dict] = [
    {
        "name": "capital_of_georgia",
        "paraphrases": [
            "საქართველოს დედაქალაქია — ",
            "საქართველოს დედაქალაქია: ",
            "საქართველოს დედაქალაქია ___.",
            "რომელი ქალაქია საქართველოს დედაქალაქი? მოკლე პასუხი.",
        ],
        "answers": {"თბილისი", "ქ. თბილისი", "tbilisi", "თბილის"},
    },
    {
        "name": "currency_of_georgia",
        "paraphrases": [
            "საქართველოს ვალუტაა — ",
            "საქართველოს ეროვნული ვალუტაა: ",
            "საქართველოს ფულის ერთეული — ",
        ],
        "answers": {"ლარი", "GEL", "ლარი (GEL)", "lari"},
    },
    {
        "name": "official_language",
        "paraphrases": [
            "საქართველოს ოფიციალური ენაა — ",
            "რა ენაა სახელმწიფო საქართველოში? მხოლოდ ერთი სიტყვა.",
            "საქართველოს სახელმწიფო ენა — ",
        ],
        "answers": {"ქართული", "ქართული ენა"},
    },
    {
        "name": "author_vefxistqavosani",
        "paraphrases": [
            "„ვეფხისტყაოსნის“ ავტორია — ",
            "ვინ დაწერა „ვეფხისტყაოსანი“? მოკლე პასუხი.",
            "ავტორი: ვეფხისტყაოსანი — ",
        ],
        "answers": {"შოთა რუსთაველი", "რუსთაველი", "შ. რუსთაველი"},
    },
    {
        "name": "author_eugene_onegin",
        "paraphrases": [
            "ვინ დაწერა „ევგენი ონეგინი“? მოკლე პასუხი.",
            "„ევგენი ონეგინის“ ავტორია — ",
            "რომელია „ევგენი ონეგინის“ ავტორი? ერთი სიტყვით.",
        ],
        "answers": {"ალექსანდრე პუშკინი", "ალექსანდრ პუშკინი", "პუშკინი",
                    "ალექსანდრ სერგეევის ძე პუშკინი", "알ექსანდრე სერგეევის ძე პუშკინი".replace("알", "ა")},
    },
    {
        "name": "anthem_of_georgia",
        "paraphrases": [
            "საქართველოს ჰიმნია — ",
            "რა ჰქვია საქართველოს ეროვნულ ჰიმნს? მოკლე პასუხი.",
            "საქართველოს ეროვნული ჰიმნი — ",
        ],
        "answers": {"თავისუფლება", "tavisupleba"},
    },
    {
        "name": "continent_region",
        "paraphrases": [
            "საქართველო მდებარეობს — ",
            "რომელ რეგიონში მდებარეობს საქართველო? მოკლე პასუხი.",
            "ეპასუხე მხოლოდ ერთი სიტყვით: საქართველოს მდებარეობა",
        ],
        "answers": {"კავკასიაში", "კავკასია", "ევრაზიაში", "ევრაზია"},
    },
    {
        "name": "largest_city",
        "paraphrases": [
            "საქართველოს უდიდესი ქალაქია — ",
            "რომელია საქართველოს ყველაზე დიდი ქალაქი? მოკლე პასუხი.",
            "საქართველოს ყველაზე დიდ ქალაქს ეწოდება — ",
        ],
        "answers": {"თბილისი", "ქ. თბილისი", "tbilisi"},
    },
]

In [34]:
rep = evaluate_tests(TESTS, stability_samples=0)
print(json.dumps(rep, ensure_ascii=False, indent=2))



{
  "overall_prompt_exact": 0.12,
  "overall_task_solved": 0.25,
  "overall_paraphrase_consistency": 0.6042,
  "task_hits": {
    "capital_of_georgia": true,
    "currency_of_georgia": false,
    "official_language": true,
    "author_vefxistqavosani": false,
    "author_eugene_onegin": false,
    "anthem_of_georgia": false,
    "continent_region": false,
    "largest_city": false
  },
  "task_consistency": {
    "capital_of_georgia": 0.5,
    "currency_of_georgia": 0.333,
    "official_language": 0.333,
    "author_vefxistqavosani": 0.667,
    "author_eugene_onegin": 1.0,
    "anthem_of_georgia": 1.0,
    "continent_region": 0.333,
    "largest_city": 0.667
  },
  "examples": [
    {
      "task": "capital_of_georgia",
      "prompt": "საქართველოს დედაქალაქია — ",
      "pred": "საქართველო და",
      "ok": false
    },
    {
      "task": "capital_of_georgia",
      "prompt": "საქართველოს დედაქალაქია: ",
      "pred": "თბილისი და",
      "ok": true
    },
    {
      "task": "capital_